In [ ]:
# CELL 0: Imports, paths, login, seeds

import os
from pathlib import Path
import random
import math
import json

import torch
from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from trl import GRPOTrainer, GRPOConfig

from utils.rewards.form_reward import form_reward
from utils.rewards.meaning_reward import meaning_reward

# meter_reward is in another file; if it's not ready yet, we use a stub = 1.0
try:
    from utils.rewards.meter_reward import meter_reward
except ImportError:
    print("⚠️ utils.rewards.meter_reward not found, using stub that returns 1.0")
    def meter_reward(completions, prompts, poem_meter=None, **kwargs):
        return [1.0 for _ in completions]


# -------------------
# 💾 Paths & IDs
# -------------------

HF_TOKEN         = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN_HERE")
BASE_MODEL_ID    = "Navid-AI/Yehia-7B-preview"
HF_ADAPTER_REPO  = "Shaer-AI/Shaer-7B-v1"

TRAIN_DATASET_ID = "Shaer-AI/ashaar-training-instructions-short"
EVAL_DATASET_ID  = "Shaer-AI/ashaar-validation-instructions-short"

# RL context lengths (prompt ~= full instruction, completion = 1 bayt)
MAX_PROMPT_LEN      = 640
MAX_COMPLETION_LEN  = 128

USE_SMALL_DEBUG = False   # flip to True for cheap dry-runs

ROOT = Path.cwd().resolve()
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR_RL_MINI = MODELS_DIR / "grpo_mini_v1"
OUT_DIR_RL_FULL = MODELS_DIR / "grpo_full_v1"
OUT_DIR_RL_MINI.mkdir(parents=True, exist_ok=True)
OUT_DIR_RL_FULL.mkdir(parents=True, exist_ok=True)

print("ROOT         :", ROOT)
print("MODELS_DIR   :", MODELS_DIR)
print("OUT_DIR_RL_MINI:", OUT_DIR_RL_MINI)
print("OUT_DIR_RL_FULL:", OUT_DIR_RL_FULL)

# Path for Yehia vLLM judge (used inside meaning_reward)
# You already used this path earlier when snapshotting Yehia.
os.environ.setdefault("YEHIA_VLLM_MODEL_PATH", "/workspace/Yehia-7B-preview-proper")

# -------------------
# 🔐 HF login + seeds
# -------------------

if not HF_TOKEN or HF_TOKEN.startswith("YOUR_"):
    raise ValueError("Please set HF_TOKEN (env or inline) before running GRPO notebook.")

login(token=HF_TOKEN)
print("✓ Logged in to Hugging Face")

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("✓ Seeds set:", SEED)

if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current device    :", torch.cuda.current_device())
    print("Device name       :", torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print("⚠️ No CUDA detected, GRPO will be unusably slow.")


In [ ]:
# CELL 1: Load datasets and build RL view with "prompt" column

if USE_SMALL_DEBUG:
    train_split = "train[:4000]"
    eval_split  = "train[:1000]"
else:
    train_split = "train"
    eval_split  = "train[:20000]"

print(f"Loading train split: {TRAIN_DATASET_ID}::{train_split}")
train_raw = load_dataset(TRAIN_DATASET_ID, split=train_split)

print(f"Loading eval split : {EVAL_DATASET_ID}::{eval_split}")
eval_raw  = load_dataset(EVAL_DATASET_ID,  split=eval_split)

print("Train size:", len(train_raw))
print("Eval  size:", len(eval_raw))

print("\nTrain sample keys:", train_raw.column_names)


def to_rl_example(ex):
    """
    Build the RL-friendly view:
      - prompt: list of (system+user) messages (no assistant)
      - poem_meter / poem_description: for rewards
      - target_verse_clean: we keep gold bayt for reward sanity tests
    """
    msgs = ex["messages"]
    msgs_no_assistant = [m for m in msgs if m.get("role") != "assistant"]

    return {
        "prompt": msgs_no_assistant,
        "poem_meter": ex.get("poem_meter"),
        "poem_description": ex.get("poem_description"),
        "target_verse_clean": ex.get("target_verse_clean"),
    }


rl_train = train_raw.map(to_rl_example, remove_columns=train_raw.column_names)
rl_eval  = eval_raw.map(to_rl_example,  remove_columns=eval_raw.column_names)

print("\nRL train columns:", rl_train.column_names)
print("RL eval  columns:", rl_eval.column_names)

print("\nExample RL row:")
ex0 = rl_train[0]
for k, v in ex0.items():
    preview = str(v)
    if isinstance(v, str):
        preview = v[:120].replace("\n", " ") + ("..." if len(v) > 120 else "")
    print(f"- {k}: {preview}")


In [ ]:
# CELL 2: Tokenizer + 4-bit Yehia + attach SFT adapter for RL

if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    compute_dtype = torch.bfloat16 if major >= 8 else torch.float16
else:
    compute_dtype = torch.float32

print("Using compute dtype:", compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# GRPO requires left padding
tokenizer.padding_side = "left"

print("✓ Tokenizer loaded.")
print("  pad_token    :", tokenizer.pad_token)
print("  padding_side :", tokenizer.padding_side)
print("  vocab_size   :", tokenizer.vocab_size)

print("\nLoading base Yehia (4-bit) for RL...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = False
print("✓ Base Yehia loaded.")

print(f"\nAttaching SFT LoRA adapter from HF repo: {HF_ADAPTER_REPO}")
model_rl = PeftModel.from_pretrained(
    base_model,
    HF_ADAPTER_REPO,
    is_trainable=True,   # RL will keep LoRA trainable
)
model_rl.config.use_cache = False
print("✓ LoRA adapter attached.")

def count_params(model):
    total = 0
    trainable = 0
    for p in model.parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
    return trainable, total

trainable, total = count_params(model_rl)
print("\nAfter RL model setup:")
print(f"- Trainable params: {trainable/1e6:.2f}M")
print(f"- Total params    : {total/1e9:.2f}B")

try:
    print("\nPEFT trainable summary:")
    model_rl.print_trainable_parameters()
except Exception as e:
    print("Could not print PEFT summary:", e)


In [ ]:
# CELL 3: Sanity check reward functions on gold target_verse_clean

N_SANITY = 8
if len(rl_eval) < N_SANITY:
    N_SANITY = len(rl_eval)

indices = random.sample(range(len(rl_eval)), k=N_SANITY)
sanity_batch = rl_eval.select(indices)

completions = sanity_batch["target_verse_clean"]
prompts_for_rewards = sanity_batch["prompt"]
meters = sanity_batch["poem_meter"]
descs  = sanity_batch["poem_description"]

print("Running form_reward on gold verses...")
form_scores = form_reward(completions, prompts_for_rewards)

print("Running meter_reward (stub or real) on gold verses...")
meter_scores = meter_reward(completions, prompts_for_rewards, poem_meter=meters)

print("Running meaning_reward (Yehia+vLLM judge) on gold verses...")
meaning_scores = meaning_reward(completions, prompts_for_rewards, poem_description=descs)

def summarize_scores(name, scores):
    s = [x for x in scores if x is not None]
    if not s:
        print(f"- {name}: all None?")
        return
    print(f"- {name}: min={min(s):.3f}, max={max(s):.3f}, mean={sum(s)/len(s):.3f}")

print("\n=== Reward sanity stats on gold data ===")
summarize_scores("form", form_scores)
summarize_scores("meter", meter_scores)
summarize_scores("meaning", meaning_scores)

print("\nSome examples:")
for i in range(N_SANITY):
    print("\n------------------------------")
    print(f"idx = {indices[i]}")
    print("meter:", meters[i])
    print("desc :", (descs[i] or "")[:200].replace("\n", " ") + "...")
    print("bayt :", completions[i])
    print("form_score   :", form_scores[i])
    print("meter_score  :", meter_scores[i])
    print("meaning_score:", meaning_scores[i])


In [ ]:
# CELL 4: Mini GRPO config + trainer on a small subset

supports_bf16 = (
    torch.cuda.is_available()
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)

print("bf16 support:", supports_bf16)

# Tiny subsets for quick sanity runs
if USE_SMALL_DEBUG:
    rl_train_mini = rl_train
    rl_eval_mini  = rl_eval
else:
    max_train = min(4096, len(rl_train))
    max_eval  = min(512, len(rl_eval))
    rl_train_mini = rl_train.select(range(max_train))
    rl_eval_mini  = rl_eval.select(range(max_eval))

print("Mini RL train size:", len(rl_train_mini))
print("Mini RL eval  size:", len(rl_eval_mini))

# Effective batch = per_device_bs * grad_accum; must be divisible by num_generations
PER_DEVICE_BS_MINI = 2
GRAD_ACCUM_MINI    = 4
NUM_GENERATIONS    = 4

grpo_config_mini = GRPOConfig(
    output_dir=str(OUT_DIR_RL_MINI),
    num_train_epochs=1.0,
    max_steps=200,  # small cap for smoke test
    per_device_train_batch_size=PER_DEVICE_BS_MINI,
    per_device_eval_batch_size=PER_DEVICE_BS_MINI,
    gradient_accumulation_steps=GRAD_ACCUM_MINI,
    learning_rate=5e-6,
    warmup_ratio=0.03,
    logging_steps=5,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",      # no saving for mini run
    bf16=supports_bf16,
    fp16=not supports_bf16,
    remove_unused_columns=False,  # we need poem_meter / poem_description for rewards
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,
    num_generations=NUM_GENERATIONS,
    temperature=0.7,
    top_p=0.9,
    loss_type="dapo",
    # no vLLM for generation here; only meaning_reward uses vLLM as judge
    use_vllm=False,
)

print("\nMini GRPO config:")
print(grpo_config_mini)

trainer_mini = GRPOTrainer(
    model=model_rl,
    reward_funcs=[form_reward, meter_reward, meaning_reward],
    args=grpo_config_mini,
    train_dataset=rl_train_mini,
    eval_dataset=rl_eval_mini,
    processing_class=tokenizer,   # tokenizer with left padding
)

print("✓ Mini GRPOTrainer ready.")


In [ ]:
# CELL 5: Run mini GRPO training

print("Starting MINI GRPO training (smoke test)...")

mini_result = trainer_mini.train()

print("\nMini GRPO training done.")
print(mini_result)

print("\nLast few log history entries:")
for entry in trainer_mini.state.log_history[-10:]:
    print(entry)


In [ ]:
# CELL 6: Full GRPO config + trainer (for real run later)

# You can tweak these once you see mini GRPO behavior.
PER_DEVICE_BS_FULL = 4
GRAD_ACCUM_FULL    = 16
NUM_GENERATIONS    = 4

grpo_config_full = GRPOConfig(
    output_dir=str(OUT_DIR_RL_FULL),
    num_train_epochs=1.0,
    max_steps=-1,  # use full 1 epoch
    per_device_train_batch_size=PER_DEVICE_BS_FULL,
    per_device_eval_batch_size=PER_DEVICE_BS_FULL,
    gradient_accumulation_steps=GRAD_ACCUM_FULL,
    learning_rate=3e-6,
    warmup_ratio=0.03,
    logging_steps=20,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    bf16=supports_bf16,
    fp16=not supports_bf16,
    remove_unused_columns=False,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,
    num_generations=NUM_GENERATIONS,
    temperature=0.7,
    top_p=0.9,
    loss_type="dapo",
    # still no vLLM for generation; judge uses its own vLLM instance
    use_vllm=False,
    # optional hub push
    push_to_hub=False,   # set True + hub_model_id if you want to push RL adapter
)

print("\nFull GRPO config:")
print(grpo_config_full)

trainer_full = GRPOTrainer(
    model=model_rl,
    reward_funcs=[form_reward, meter_reward, meaning_reward],
    args=grpo_config_full,
    train_dataset=rl_train,
    eval_dataset=rl_eval,
    processing_class=tokenizer,
)

print("✓ Full GRPOTrainer ready (do not run train() yet if you just want mini).")


In [ ]:
# CELL 7: Run FULL GRPO training (when you're ready)

print("Starting FULL GRPO training... (this may be heavy)")

full_result = trainer_full.train()

print("\nFull GRPO training done.")
print(full_result)

print("\nSaving final RL-tuned adapter...")
trainer_full.save_model(str(OUT_DIR_RL_FULL))
tokenizer.save_pretrained(str(OUT_DIR_RL_FULL))

print("\nLast few log history entries (full):")
for entry in trainer_full.state.log_history[-10:]:
    print(entry)
